In [0]:
DECLARE OR REPLACE VARIABLE std_tariff_sk STRING;
SET VAR std_tariff_sk = (SELECT sk FROM smart_meters.gold.dim_tariffs WHERE id = 1);

CREATE OR REPLACE TEMP VIEW smart_meters_test AS
-- Dados reais da MAC002765 (ToU)
SELECT * FROM smart_meters.gold.fact_consumption
WHERE dim_household_lclid = 'MAC002765'

UNION ALL

-- Dados artificiais: primeiros 3 dias com Std (tariff_id = 1)
SELECT
    0.5           AS energy_spent,
    0.07          AS money_spent,           -- 0.5 kWh × 0.14 € (Std)
    std_tariff_sk AS dim_tariff_sk,         -- Std UUID
    'MAC002765'   AS dim_household_lclid,
    202301010000  AS dim_time_id,           -- YYYYMMDDHHmm
    NULL          AS sk_weather

UNION ALL
    SELECT 0.3, 0.042, std_tariff_sk, 'MAC002765', 202301010030, NULL
UNION ALL
    SELECT 0.6, 0.084, std_tariff_sk, 'MAC002765', 202301020000, NULL
UNION ALL
    SELECT 0.4, 0.056, std_tariff_sk, 'MAC002765', 202301020030, NULL
UNION ALL
    SELECT 0.7, 0.098, std_tariff_sk, 'MAC002765', 202301030000, NULL;


SELECT 
    dim_household_lclid, 
    COUNT(DISTINCT d.type) AS numero_de_tarifas_diferentes
FROM smart_meters_test f
JOIN smart_meters.gold.dim_tariffs d ON f.dim_tariff_sk = d.sk
GROUP BY dim_household_lclid
HAVING COUNT(DISTINCT d.type) > 1;